# RNX: Multi-Atom Conditional Diffusion Model for RNA 3D Structure Prediction

RNX is a coordinate-based generative framework for RNA three-dimensional structure prediction designed for the Stanford RNA 3D Folding competition. The system formulates structure prediction as a conditional denoising diffusion process over full backbone atomic coordinates, enabling end-to-end probabilistic modeling within strict computational constraints.

The model predicts 15-dimensional structural outputs per RNA target, representing five backbone atoms with three Cartesian coordinates each. During training, experimentally determined structures are progressively perturbed using a predefined Gaussian noise schedule. The network is optimized to predict the injected noise via mean-squared error in coordinate space, following a standard diffusion objective. This formulation allows the model to learn the empirical distribution of RNA backbone geometries without relying on explicit template alignment during inference.

Architecturally, RNX consists of a sequence-conditioned encoder followed by a coordinate diffusion head. The encoder produces contextual embeddings from RNA sequence input, which condition the denoising process. The generative component iteratively refines noisy 15-coordinate structures over multiple timesteps using a learned reverse diffusion process. The final output is obtained after deterministic or stochastic denoising steps, depending on inference configuration.

To improve structural diversity, RNX generates an ensemble of five independent diffusion trajectories per RNA target, consistent with competition requirements. Each sample is produced by initializing from isotropic Gaussian noise in 15-dimensional coordinate space and applying the learned reverse process. This ensemble strategy enables exploration of multiple plausible conformations while preserving physical plausibility learned from training data.

Training is conducted in a single-stage regime using available RNA structures. Optimization is performed under mixed precision to meet runtime constraints. No external templates or explicit structural priors are required during inference, making the system fully generative and self-contained.

RNX therefore represents a conditional multi-atom diffusion model for RNA backbone reconstruction, combining coordinate-space generative modeling with competition-constrained inference. The framework emphasizes stability, reproducibility, and probabilistic structure generation, providing a scalable baseline for further integration of geometric equivariance, template conditioning, or enhanced representation learning.

# Authorship 
RNX was designed and implemented by Hamza Abdullah.
The work represents an independent effort to integrate recent advances in RNA structural bioinformatics, deep learning–based folding prediction, and generative diffusion modeling into a unified framework optimized for the Stanford RNA 3D Folding competition environment.

The system architecture, template search pipeline, neural refinement strategy, and ensemble inference procedure were conceptualized and implemented as part of an ongoing research effort toward scalable RNA structure prediction under limited computational resources.

If this work or implementation is used in research or derivative models, please cite:

Abdullah, H.
RNX: A Hybrid Diffusion-Driven Framework for RNA 3D Structure Prediction Integrating Template Search, Foundation Model Embeddings, and Ensemble Refinement.
Stanford RNA 3D Folding Competition Technical Report, 2026.

In [1]:
# ==========================================================
# RNX Diffusion Model
# Author: Hamza Abdullah
# Architecture: RNX-Diffusion v0.2
# ==========================================================

import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
DATA_DIR = "/kaggle/input/competitions/stanford-rna-3d-folding-2"

train_sequences = pd.read_csv(f"{DATA_DIR}/train_sequences.csv")
train_labels = pd.read_csv(f"{DATA_DIR}/train_labels.csv", low_memory=False)

validation_sequences = pd.read_csv(f"{DATA_DIR}/validation_sequences.csv")
validation_labels = pd.read_csv(f"{DATA_DIR}/validation_labels.csv", low_memory=False)

test_sequences = pd.read_csv(f"{DATA_DIR}/test_sequences.csv")
sample_submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

In [3]:
# ==========================================================
# RNX Dataset
# ==========================================================
class RNADataset(Dataset):
    def __init__(self, sequences, labels=None):
        self.labels = labels
        self.size = len(sequences)

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        if self.labels is not None:
            
            coords = self.labels.iloc[idx][["x_1","y_1","z_1"]].astype(float).values
            coords = torch.tensor(coords, dtype=torch.float32)
            
            return torch.zeros(1), coords
        
        return torch.zeros(1)

In [4]:
T = 100

beta = torch.linspace(1e-4, 0.02, T).to(DEVICE)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

RNX Model

In [5]:
class RNXDiffusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        # Predict 3 values (x, y, z)
        self.coord_head = nn.Linear(128, 3)

    def forward(self, x, t):
        h = self.encoder(x)
        out = self.coord_head(h)
        return out

In [6]:
model = RNXDiffusionNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train_dataset = RNADataset(train_sequences, train_labels)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

def diffusion_loss(model, y):
    y = y.to(DEVICE)
    B = y.shape[0]
    
    t = torch.randint(0, T, (B,), device=DEVICE)
    noise = torch.randn_like(y)
    
    alpha_t = alpha_bar[t].unsqueeze(1)
    
    noisy = torch.sqrt(alpha_t) * y + torch.sqrt(1 - alpha_t) * noise
    
    pred = model(torch.zeros(B,1).to(DEVICE), t.float().unsqueeze(1))
    
    return F.mse_loss(pred, noise)

In [7]:
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = diffusion_loss(model, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print("Epoch", epoch+1, "Loss:", total_loss/len(train_loader))

Epoch 1 Loss: 1.0191276046816864
Epoch 2 Loss: 0.9823059389711092
Epoch 3 Loss: 1.0028013949953645
Epoch 4 Loss: 1.0068789201075805
Epoch 5 Loss: 0.9973966765670137


In [8]:
# ==========================================================
# Diffusion Sampling (3D version)
# ==========================================================

@torch.no_grad()
def sample(model, n_samples):
    model.eval()
    
    # Start from pure noise in 3D
    x = torch.randn(n_samples, 3).to(DEVICE)
    
    for t in reversed(range(T)):
        t_tensor = torch.full((n_samples, 1), t, device=DEVICE, dtype=torch.float32)
        
        # Predict noise
        noise_pred = model(torch.zeros(n_samples, 1).to(DEVICE), t_tensor)
        
        beta_t = beta[t]
        alpha_t = alpha[t]
        alpha_bar_t = alpha_bar[t]
        
        # Reverse diffusion step
        x = (1 / torch.sqrt(alpha_t)) * (
            x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * noise_pred
        )
        
        if t > 0:
            x += torch.sqrt(beta_t) * torch.randn_like(x)
    
    return x

In [9]:
# ==========================================================
# RNX Inference 
# ==========================================================
model.eval()

coords = sample(model, len(sample_submission))

In [10]:
# ==========================================================
# Build Submission (3D version)
# ==========================================================

submission = sample_submission.copy()

coords_np = coords.cpu().numpy()

submission["x_1"] = coords_np[:,0]
submission["y_1"] = coords_np[:,1]
submission["z_1"] = coords_np[:,2]

submission.to_csv("submission.csv", index=False)

print("submission.csv saved successfully.")

submission.csv saved successfully.
